In [ ]:
import torch, torch.nn as nn

# --- CA utils (periodic, vectorized) ---
def rule_table(rule):
    return torch.tensor([(rule >> i) & 1 for i in range(8)], dtype=torch.uint8)

def evolve_once(state, tbl):  # state: [B,N] in {0,1}
    L = torch.roll(state,  1, dims=-1)
    R = torch.roll(state, -1, dims=-1)
    idx = ((L << 2) | (state << 1) | R).long()  # <-- make it Long
    return tbl[idx]  # shape [B,N], dtype stays as tbl's dtype

def jump_ahead(state, tbl, H):
    for _ in range(H):
        state = evolve_once(state, tbl)
    return state

# --- Dataset: pairs (s^t -> s^{t+H}) ---
def make_batch(B, N, rule, H, device="cpu"):
    x = torch.randint(0, 2, (B, N), dtype=torch.long, device=device)
    y = jump_ahead(x.clone(), rule_table(rule).to(device), H)
    # to float logits targets in {0,1}
    return x.unsqueeze(1).float(), y.unsqueeze(1).float()

# --- Shallow model: receptive field >= 2H+1 ---
class ShallowCNN(nn.Module):
    def __init__(self, H, hidden=16):
        super().__init__()
        k = 2*H + 1
        self.conv1 = nn.Conv1d(1, hidden, kernel_size=k, padding=k//2,
                               padding_mode='circular', bias=False)
        self.out = nn.Conv1d(hidden, 1, kernel_size=1, bias=True)
        self.act = nn.ReLU() #nn.Identity()
    def forward(self, x):
        return self.out(self.act(self.conv1(x)))

# --- Tiny training loop ---
def train_rule(rule, H=16, N=256, steps=500, B=64, device="cpu"):
    model = ShallowCNN(H).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.BCEWithLogitsLoss()

    for step in range(steps):
        x, y = make_batch(B, N, rule, H, device)
        logits = model(x)
        loss = loss_fn(logits, y)
        opt.zero_grad(); loss.backward(); opt.step()
        if (step+1) % 25 == 0:
            with torch.no_grad():
                acc = ((logits.sigmoid() > 0.5) == (y > 0.5)).float().mean().item()
            print(f"rule {rule:3d} | H={H:2d} | step {step+1:3d} | loss {loss:.3f} | acc {acc:.3f}")


In [73]:
# Parameters:
H = 128

In [74]:

# Example runs:
train_rule(150, H=H)   # should climb to ~1.00 quickly


rule 150 | H=128 | step  25 | loss 0.693 | acc 0.512
rule 150 | H=128 | step  50 | loss 0.677 | acc 0.620
rule 150 | H=128 | step  75 | loss 0.660 | acc 0.789
rule 150 | H=128 | step 100 | loss 0.635 | acc 0.966
rule 150 | H=128 | step 125 | loss 0.604 | acc 0.988
rule 150 | H=128 | step 150 | loss 0.561 | acc 1.000
rule 150 | H=128 | step 175 | loss 0.512 | acc 1.000
rule 150 | H=128 | step 200 | loss 0.459 | acc 1.000
rule 150 | H=128 | step 225 | loss 0.405 | acc 1.000
rule 150 | H=128 | step 250 | loss 0.351 | acc 1.000
rule 150 | H=128 | step 275 | loss 0.301 | acc 1.000
rule 150 | H=128 | step 300 | loss 0.256 | acc 1.000
rule 150 | H=128 | step 325 | loss 0.217 | acc 1.000
rule 150 | H=128 | step 350 | loss 0.185 | acc 1.000
rule 150 | H=128 | step 375 | loss 0.158 | acc 1.000
rule 150 | H=128 | step 400 | loss 0.135 | acc 1.000
rule 150 | H=128 | step 425 | loss 0.117 | acc 1.000
rule 150 | H=128 | step 450 | loss 0.100 | acc 1.000
rule 150 | H=128 | step 475 | loss 0.087 | acc

In [75]:
# Example runs:
train_rule(90, H=H)   # should also get ~1.00

rule  90 | H=128 | step  25 | loss 0.011 | acc 1.000
rule  90 | H=128 | step  50 | loss 0.003 | acc 1.000
rule  90 | H=128 | step  75 | loss 0.002 | acc 1.000
rule  90 | H=128 | step 100 | loss 0.001 | acc 1.000
rule  90 | H=128 | step 125 | loss 0.001 | acc 1.000
rule  90 | H=128 | step 150 | loss 0.001 | acc 1.000
rule  90 | H=128 | step 175 | loss 0.001 | acc 1.000
rule  90 | H=128 | step 200 | loss 0.001 | acc 1.000
rule  90 | H=128 | step 225 | loss 0.001 | acc 1.000
rule  90 | H=128 | step 250 | loss 0.000 | acc 1.000
rule  90 | H=128 | step 275 | loss 0.000 | acc 1.000
rule  90 | H=128 | step 300 | loss 0.000 | acc 1.000
rule  90 | H=128 | step 325 | loss 0.000 | acc 1.000
rule  90 | H=128 | step 350 | loss 0.000 | acc 1.000
rule  90 | H=128 | step 375 | loss 0.000 | acc 1.000
rule  90 | H=128 | step 400 | loss 0.000 | acc 1.000
rule  90 | H=128 | step 425 | loss 0.000 | acc 1.000
rule  90 | H=128 | step 450 | loss 0.000 | acc 1.000
rule  90 | H=128 | step 475 | loss 0.000 | acc

In [76]:
# Example runs:

train_rule( 30, H=H)   # should stall far below 1.00; gets worse as H increases

rule  30 | H=128 | step  25 | loss 0.695 | acc 0.497
rule  30 | H=128 | step  50 | loss 0.694 | acc 0.499
rule  30 | H=128 | step  75 | loss 0.694 | acc 0.491
rule  30 | H=128 | step 100 | loss 0.693 | acc 0.501
rule  30 | H=128 | step 125 | loss 0.693 | acc 0.501
rule  30 | H=128 | step 150 | loss 0.693 | acc 0.502
rule  30 | H=128 | step 175 | loss 0.693 | acc 0.503
rule  30 | H=128 | step 200 | loss 0.693 | acc 0.497
rule  30 | H=128 | step 225 | loss 0.693 | acc 0.503
rule  30 | H=128 | step 250 | loss 0.693 | acc 0.504
rule  30 | H=128 | step 275 | loss 0.693 | acc 0.498
rule  30 | H=128 | step 300 | loss 0.693 | acc 0.500
rule  30 | H=128 | step 325 | loss 0.693 | acc 0.496
rule  30 | H=128 | step 350 | loss 0.693 | acc 0.499
rule  30 | H=128 | step 375 | loss 0.694 | acc 0.492
rule  30 | H=128 | step 400 | loss 0.694 | acc 0.495
rule  30 | H=128 | step 425 | loss 0.693 | acc 0.495
rule  30 | H=128 | step 450 | loss 0.693 | acc 0.499
rule  30 | H=128 | step 475 | loss 0.693 | acc